Plan is the following

1. Trying the PS on binomial Gaussuan in D- dimensions $(D=10, \, 10^2, \, 10^3, \, 10^4)$ for a target distribution 

$p(\theta | y) = \frac{1}{3} \mathcal{N}(0, 1) + \frac{2}{3} \mathcal{N}(\sqrt{D}, 1 / 2)$.

with prior $\pi(\theta) = \mathcal{U}(\theta | -4\sqrt{D}, 4\sqrt{D})$

and the Likelihood function $\mathcal{L}(\theta) = p(y| \theta) =  \frac{1}{3} \mathcal{N}(0, 1) + \frac{2}{3} \mathcal{N}(\sqrt{D}, 1 / 2)$

Minas propeses trying PS in combination with HMC/LMC(ULA, MALA)/NUTS/MCLMC/etc and compare that to: SMC and perhaps Parallel Tempering.

maby if there is time I could also try: ULA, diagonal preconditioned ULA, diagonal preconditioned HMC or MALA, low-rank / dense adaptive mass matrix

Quality evaluation 

I will use the same metrices as is done in PS paper, by calculating $MSE$ for:
- bias and variance of first $f_{1}(\theta) = \theta$ and second $f_{2}(\theta) = \theta^2$ posterior moment.
- log marginal likelihood estimate $\log \mathcal{Z}$ 

for $\log \mathcal{Z}$ the MSE is computed across $L=100$ independent runs

$$MSE_{\log \mathcal{Z}} = \frac{1}{L} \sum_{l=1}^{L} (\log \mathcal{Z}_l - \log \mathcal{Z})^2,$$

the $\log \mathcal{Z}$ is a reference value computed from 100 independent runs of $10^{4}$ particles SMC, where we set value to $\alpha = 0.999$ (this paramether has a role in the adaptive tempering schedule, controlling the ESS threshold).

again same as in the PS paper, for D-dim posterior moments I calculated the maximum squared bias across dimensions:

$$ b_{i}^{2} = \max_{d} ( \frac{\hat{f}_{i,d} - \mu (f_{i,d})}{\sigma (f_{i,d})} )^{2} $$

where $\hat{f}_{i,d}$ is the estimated coordiate - wise mean across $L=100$ independent runs, while $\mu(f_{i,d})$, $\sigma(f_{i,d})$ are the mean and standard deviation of the coordinate-wise posterior moment $f_{i,d}$ computed from 100 independent runs of $10^{4}$ particles SMC.

additional metrices to keep an eye on: ESS per second, ESS per dim min/median, ESS/gradient eval, MSE of $\theta$ and $\theta^{2}$, gradient evaluation counts, variance of log weights,acceptance rate (kernel), number of resampling steps, log evidence variance, ESJD, ESS trajectory over anenealing steps, weight entropy, time-to-ESS, Wasserstein distance or KSD


projects structure:
```text
code/
├── targets.py          # prior, likelihood, exact/analytic moments if available
├── samplers.py         # wrappers for PS+HMC, PS+ HMC/LMC/NUTS/MCLMC/etc
├── metrics.py          # one-run metrics and multi-run aggregations
├── reference.py        # builds/stores reference moments and reference logZ
├── experiment.py       # loops over D, sampler, kernel, seeds
└── plots.py            # plotting of results to compare the kernels
```

**targets.py**

For now I have only Multymodal Gauss experiment implemented. Here I define the target distribution, the prior, the likelihood, posterior and how to sample initial particles form the posterior. 

We have a Target dataclass that combines the prior, likelihood, posterior, and prior-sampling functions. In the current experiment, the target is a 
D-dimensional bimodal Gaussian mixture likelihood combined with a uniform-box prior 
$[- 4 \sqrt{D},  4 \sqrt{D}]^{D}$. For the sanity check we als o provide the approximate mixture-based formulas for the posterior mean and second moment.

**reference.py** 

We have an adaptive tempered (how we update temperature) Sequential Monte Carlo sampler. It uses systematic resampling and a Random-Walk Metropolis move kernel. The random-walk proposal covariance is estimated from the particle cloud, and the proposal scale is tuned by Robbins-Monro adaptation toward a target acceptance rate. It estimates posterior moments and the log evidence. 
The code is paralelised to run many (num\_workers) seeds simultaniously using CPU cores, later aggregating the results into a stable reference estimate.

the code does not yet support: ESJD, weight entropy, Wasserstein, KSD, gradient-eval counts, ESS/gradient eval, variance of log weights, explicit resampling counts

**samplers.py** 

for each of the cobinations PS + RW/HMC/LMC(ULA + MALA)/NUTS/MCLMC/etc calculate: 
logZ
posterior_mean
posterior_second_moment
final_ess
acceptance_rate_mean
n_iter
runtime_sec
tempering_path
logZ_path
ess_path
acceptance_path
elapsed_time_path
gradient_eval_count
resampling_steps
variance_log_weights_path
weight_entropy_path
esjd_path

the PS has to be adaptive tempered as it is defined in the PS paper [see page 5]. 





In [ ]:
import numpy as np
# setup
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import jax
import jax.numpy as jnp
import blackjax
from blackjax.smc.persistent_sampling import compute_persistent_ess
from blackjax.smc.resampling import systematic
from blackjax.smc import extend_params

# linear temepering schedule
def log_prior(theta: jnp.ndarray, d: int) -> jnp.ndarray:
    bound = jnp.sqrt(float(d))
    logpdf_coords = jax.scipy.stats.uniform.logpdf(
        theta,
        loc=-bound,
        scale=2.0 * bound,
    )
    return jnp.sum(logpdf_coords)

def log_gaussian_diag(x: jnp.ndarray, mean: jnp.ndarray, var: float) -> jnp.ndarray:
    return -0.5 * (
        jnp.sum((x - mean) ** 2 / var)
        + x.shape[-1] * jnp.log(2.0 * jnp.pi * var)
    )

def log_likelihood(theta: jnp.ndarray, d: int) -> jnp.ndarray:
    mean1 = jnp.zeros(d)
    mean2 = jnp.sqrt(float(d)) * jnp.ones(d)

    log_prob1 = log_gaussian_diag(theta, mean1, 1.0)
    log_prob2 = log_gaussian_diag(theta, mean2, 0.5)

    return jax.scipy.special.logsumexp(
        jnp.array([log_prob1, log_prob2]),
        b=jnp.array([1/3, 2/3]),
    )

def log_posterior(theta: jnp.ndarray, d: int) -> jnp.ndarray:
    return log_prior(theta, d) + log_likelihood(theta, d)

# dim=10, t= 3.2sec
dimension = 100

# SMC configuration
num_particles = 1000

log_prior_fn = lambda theta: log_prior(theta, dimension)
log_likelihood_fn = lambda theta: log_likelihood(theta, dimension)
log_posterior_fn = lambda theta: log_posterior(theta, dimension)



# define fixed linear tempering schedule
num_temperatures = 100
tempering_schedule = jnp.linspace(0.0, 1.0, num_temperatures)
# Set random seed
key = jax.random.PRNGKey(42)

key, init_key = jax.random.split(key)
initial_particles = jax.random.uniform( init_key,
                                        shape = (num_particles, dimension),
                                        minval = -jnp.sqrt(jnp.array([dimension])), 
                                        maxval = jnp.sqrt(jnp.array([dimension])), )

# HMC parameters for MCMC moves
hmc_parameters = { "step_size": 0.02,
                    "inverse_mass_matrix": jnp.ones(dimension), 
                    "num_integration_steps": 100, }


# create Persistent Sampling kernel
ps_kernel = blackjax.persistent_sampling_smc(   logprior_fn=log_prior_fn,
                                                loglikelihood_fn=log_likelihood_fn,
                                                n_schedule=num_temperatures,
                                                mcmc_step_fn=blackjax.hmc.build_kernel(), # HMC 
                                                mcmc_init_fn=blackjax.hmc.init, 
                                                mcmc_parameters=extend_params(hmc_parameters),
                                                resampling_fn=systematic,
                                                num_mcmc_steps=25, )

# create inference loop
def inference_loop(key, kernel, initial_state, schedule): # SMC inference loop 

    def one_step(carry, lmbda):
        key, state = carry
        key, step_key = jax.random.split(key)
        new_state, _ = kernel.step(step_key, state, lmbda)
        ess = compute_persistent_ess( jnp.log(new_state.persistent_weights), normalize_weights=True )
        return (key, new_state), ess

    init_state = kernel.init(initial_state)
    (_, final_state), ess_history = jax.lax.scan(one_step, (key, init_state), schedule)
    return final_state, ess_history

key, ps_key = jax.random.split(key)
ps_final_state, ess_history = inference_loop(
    ps_key,
    ps_kernel,
    initial_particles,
    tempering_schedule,
)

ps_ess = compute_persistent_ess(
    jnp.log(ps_final_state.persistent_weights), normalize_weights=True
)
print(f"Persistent Sampling Results:")
print(
    f"  Final ESS: {ps_ess:.1f} (from {num_particles * num_temperatures} persistent particles, {ps_ess/(num_particles * num_temperatures):.2%})"
)
print(f"  Log evidence estimate: {ps_final_state.log_Z:.2f}\n")

Persistent Sampling Results:
  Final ESS: 15267.5 (from 100000 persistent particles, 15.27%)
  Log evidence estimate: -301.64



In [ ]:
#β_t is computed internally from the ESS condition using bisection

import jax
import jax.numpy as jnp
import blackjax
import blackjax.smc.resampling as resampling
from blackjax.smc import extend_params

dimension = 15

def log_prior(theta, d):
    bound = jnp.sqrt(float(d))
    logpdf_coords = jax.scipy.stats.uniform.logpdf(
        theta, loc=-bound, scale=2.0 * bound
    )
    return jnp.sum(logpdf_coords)

def log_gaussian_diag(x, mean, var):
    return -0.5 * (
        jnp.sum((x - mean) ** 2 / var)
        + x.shape[-1] * jnp.log(2.0 * jnp.pi * var)
    )

def log_likelihood(theta, d):
    mean1 = jnp.zeros(d)
    mean2 = jnp.sqrt(float(d)) * jnp.ones(d)

    log_prob1 = log_gaussian_diag(theta, mean1, 1.0)
    log_prob2 = log_gaussian_diag(theta, mean2, 0.5)

    return jax.scipy.special.logsumexp(
        jnp.array([log_prob1, log_prob2]),
        b=jnp.array([1.0 / 3.0, 2.0 / 3.0]),
    )

log_prior_fn = lambda theta: log_prior(theta, dimension)
log_likelihood_fn = lambda theta: log_likelihood(theta, dimension)

num_particles = 32
max_iterations = 100
alpha = 0.999   # paper notation

key = jax.random.PRNGKey(42)
key, init_key = jax.random.split(key)

bound = jnp.sqrt(float(dimension))
initial_particles = jax.random.uniform(
    init_key,
    shape=(num_particles, dimension),
    minval=-bound,
    maxval=bound,
)

hmc_parameters = {
    "step_size": 0.001,
    "inverse_mass_matrix": jnp.ones(dimension),
    "num_integration_steps": 20,
}

adaptive_ps_kernel = blackjax.adaptive_persistent_sampling_smc(
    logprior_fn=log_prior_fn,
    loglikelihood_fn=log_likelihood_fn,
    max_iterations=max_iterations,
    mcmc_step_fn=blackjax.hmc.build_kernel(),
    mcmc_init_fn=blackjax.hmc.init,
    mcmc_parameters=extend_params(hmc_parameters),
    resampling_fn=resampling.systematic,
    target_ess=alpha,
    num_mcmc_steps=25,
)

def adaptive_loop(key, kernel, initial_particles, max_iterations):
    init_state = kernel.init(initial_particles)

    def cond_fun(carry): # stopping rule
        i, state, _ = carry
        return (state.tempering_param < 1.0) & (i < max_iterations)

    def body_fun(carry):
        i, state, key = carry
        key, subkey = jax.random.split(key)
        new_state, info = kernel.step(subkey, state) # new step is internaly calculated 
        return i + 1, new_state, key

    n_iter, final_state, _ = jax.lax.while_loop(
        cond_fun,
        body_fun,
        (0, init_state, key),
    )
    return n_iter, final_state

key, run_key = jax.random.split(key)
n_iter, ps_final_state = adaptive_loop(
    run_key,
    adaptive_ps_kernel,
    initial_particles,
    max_iterations,
)

# optional: remove unused padding
ps_final_state = blackjax.persistent_sampling.remove_padding(ps_final_state)

print("n_iter =", n_iter)
print("final beta =", ps_final_state.tempering_param)
print("log_Z =", ps_final_state.log_Z)

n_iter = 5
final beta = 1.0
log_Z = -36.844704


In [2]:
import os
os.environ["JAX_PLATFORM_NAME"] = "cpu"

import jax
print(jax.devices())

from reference import build_reference_stats_chunked

ref = build_reference_stats_chunked(
    dimension=10,
    num_particles_reference=32,
    num_reference_runs=10,
    chunk_size=5,
    max_iterations=200,
    alpha=0.9,
    num_mcmc_steps=5,
    rw_step_size=0.2,
)

print("logZ_ref_mean:", ref.logZ_ref_mean)
print("logZ_ref_std:", ref.logZ_ref_std)
print("final_ess_mean:", ref.final_ess_mean)
print("acceptance_rate_mean:", ref.acceptance_rate_mean)
print("n_iter_mean:", ref.n_iter_mean)

[CpuDevice(id=0)]
Running reference chunk 0:5
Running reference chunk 5:10
logZ_ref_mean: -10.486807537078857
logZ_ref_std: 0.02029896586745094
final_ess_mean: 31.7930419921875
acceptance_rate_mean: 0.9982144951820373
n_iter_mean: 2.0


In [ ]:
"""from reference import build_reference_stats_chunked, load_reference_stats, nan_report_reference_stats

ref = build_reference_stats_chunked(
    dimension=10,
    num_particles_reference=32,
    num_reference_runs=10,
    chunk_size=5,
    max_iterations=200,
    alpha=0.9,
    num_mcmc_steps=5,
    rw_step_size=0.5,
    checkpoint_path="../data/checkpoints/ref_dim10_ckpt.json",
    save_final_path="../data/checkpoints/ref_dim10_final.json",
)

print("logZ_ref_mean:", ref.logZ_ref_mean)
print("logZ_ref_std:", ref.logZ_ref_std)
print("final_ess_mean:", ref.final_ess_mean)
print("acceptance_rate_mean:", ref.acceptance_rate_mean)
print("n_iter_mean:", ref.n_iter_mean)

print("\nNaN report from in-memory object:")
print(nan_report_reference_stats(ref))"""

"""from reference import run_reference_sampler_once

out = run_reference_sampler_once(
    dimension=10,
    num_particles=32,
    seed=0,
    max_iterations=200,
    alpha=0.9,
    num_mcmc_steps=10,
    rw_step_size=2.0,
    rm_c=5.0,
    rm_t0=1.0,
    rm_kappa=0.6,)


print(out["acceptance_rate_mean"])
print(out["rw_scale_path"])"""



import os

# Must be set BEFORE importing jax/reference
os.environ["JAX_PLATFORM_NAME"] = "cpu"

from reference import build_reference_stats_chunked
import jax

print(jax.default_backend())
print(jax.devices())

ref = build_reference_stats_chunked(
    dimension=10,
    num_particles_reference=10,
    num_reference_runs=8,
    chunk_size=4,
    parallel=True,
    num_workers=2,
    checkpoint_path=None,
    save_final_path=None,
    max_iterations=200,
    alpha=0.9,
    num_mcmc_steps=10,
    rw_step_size=2.0,
    rm_c=5.0,
    rm_t0=1.0,
    rm_kappa=0.6,
    verbose=True,
)

ref

Platform 'METAL' is experimental and not all JAX functionality may be correctly supported!
W0000 00:00:1775597418.158995 26982358 mps_client.cc:510] WARNING: JAX Apple GPU support is experimental and not all JAX functionality is correctly supported!
I0000 00:00:1775597418.210606 26982358 service.cc:145] XLA service 0x1173fe3a0 initialized for platform METAL (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1775597418.210796 26982358 service.cc:153]   StreamExecutor device (0): Metal, <undefined>
I0000 00:00:1775597418.212678 26982358 mps_client.cc:406] Using Simple allocator.
I0000 00:00:1775597418.212708 26982358 mps_client.cc:384] XLA backend will use up to 11452858368 bytes on device 0 for SimpleAllocator.


Metal device set to: Apple M1 Pro

systemMemory: 16.00 GB
maxCacheSize: 5.33 GB

cpu
[CpuDevice(id=0)]
Running reference chunk seeds 0:4


Platform 'METAL' is experimental and not all JAX functionality may be correctly supported!
Platform 'METAL' is experimental and not all JAX functionality may be correctly supported!
W0000 00:00:1775597419.789782 26982811 mps_client.cc:510] WARNING: JAX Apple GPU support is experimental and not all JAX functionality is correctly supported!
W0000 00:00:1775597419.789816 26982809 mps_client.cc:510] WARNING: JAX Apple GPU support is experimental and not all JAX functionality is correctly supported!
I0000 00:00:1775597419.798436 26982811 service.cc:145] XLA service 0x102f926a0 initialized for platform METAL (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1775597419.798443 26982809 service.cc:145] XLA service 0x14d1e8be0 initialized for platform METAL (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1775597419.798449 26982811 service.cc:153]   StreamExecutor device (0): Metal, <undefined>
I0000 00:00:1775597419.798451 26982809 service.cc:153]   Stre

Metal device set to: Metal device set to: Apple M1 ProApple M1 Pro


systemMemory: 16.00 GB
maxCacheSize: 5.33 GB


systemMemory: 16.00 GB
maxCacheSize: 5.33 GB

Run 1/8 completed (seed=1) | elapsed=9.9s | ETA=69.6s
Run 2/8 completed (seed=0) | elapsed=10.1s | ETA=30.3s
Run 3/8 completed (seed=2) | elapsed=14.9s | ETA=24.9s
Run 4/8 completed (seed=3) | elapsed=15.2s | ETA=15.2s


I0000 00:00:1775597433.904781 26982811 mps_client.h:209] MetalClient destroyed.
I0000 00:00:1775597433.905573 26982809 mps_client.h:209] MetalClient destroyed.


Running reference chunk seeds 4:8


Platform 'METAL' is experimental and not all JAX functionality may be correctly supported!
W0000 00:00:1775597435.671961 26983075 mps_client.cc:510] WARNING: JAX Apple GPU support is experimental and not all JAX functionality is correctly supported!
Platform 'METAL' is experimental and not all JAX functionality may be correctly supported!
W0000 00:00:1775597435.672038 26983073 mps_client.cc:510] WARNING: JAX Apple GPU support is experimental and not all JAX functionality is correctly supported!
I0000 00:00:1775597435.680565 26983075 service.cc:145] XLA service 0x17256b650 initialized for platform METAL (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1775597435.680580 26983075 service.cc:153]   StreamExecutor device (0): Metal, <undefined>
I0000 00:00:1775597435.680568 26983073 service.cc:145] XLA service 0x161047180 initialized for platform METAL (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1775597435.680589 26983073 service.cc:153]   Stre

Metal device set to: Metal device set to: Apple M1 Pro
Apple M1 Pro

systemMemory: 16.00 GB
maxCacheSize: 5.33 GB


systemMemory: 16.00 GB
maxCacheSize: 5.33 GB

Run 5/8 completed (seed=4) | elapsed=24.7s | ETA=14.8s
Run 6/8 completed (seed=5) | elapsed=24.8s | ETA=8.3s
Run 7/8 completed (seed=6) | elapsed=28.6s | ETA=4.1s
Run 8/8 completed (seed=7) | elapsed=29.3s | ETA=0.0s


I0000 00:00:1775597447.986128 26983073 mps_client.h:209] MetalClient destroyed.
I0000 00:00:1775597447.997627 26983075 mps_client.h:209] MetalClient destroyed.


ReferenceStats(target_name='gaussian_mixture', dimension=10, num_reference_runs=8, num_particles_reference=10, logZ_ref_mean=-14.82721921801567, logZ_ref_std=0.27739017217749556, f1_mean_ref=array([-0.13665238, -0.11467347, -0.12024808,  0.07674778,  0.10789358,
       -0.00690053,  0.0491311 , -0.04753275,  0.11270276, -0.15135184],
      dtype=float32), f1_std_ref=array([0.2023211 , 0.2889461 , 0.22826041, 0.2122896 , 0.1046873 ,
       0.14235917, 0.16381192, 0.2811834 , 0.17379533, 0.18266797],
      dtype=float32), f2_mean_ref=array([0.36933744, 0.31083322, 0.42252302, 0.32446432, 0.3881883 ,
       0.27082714, 0.51809275, 0.34332392, 0.31191427, 0.3404752 ],
      dtype=float32), f2_std_ref=array([0.06912333, 0.30184203, 0.2565847 , 0.20131138, 0.22619368,
       0.15975286, 0.2527439 , 0.22957867, 0.23152867, 0.2419123 ],
      dtype=float32), final_ess_mean=9.861634254455566, final_ess_std=0.15848610866668783, acceptance_rate_mean=0.23911158846818753, acceptance_rate_std=0.0082

In [1]:
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor
from reference import debug_worker_backend

ctx = mp.get_context("spawn")
with ProcessPoolExecutor(max_workers=1, mp_context=ctx) as ex:
    fut = ex.submit(debug_worker_backend)
    print(fut.result())

Platform 'METAL' is experimental and not all JAX functionality may be correctly supported!


Metal device set to: Apple M1 Pro

systemMemory: 16.00 GB
maxCacheSize: 5.33 GB



W0000 00:00:1775597739.431614 26986495 mps_client.cc:510] WARNING: JAX Apple GPU support is experimental and not all JAX functionality is correctly supported!
I0000 00:00:1775597739.450444 26986495 service.cc:145] XLA service 0x16b593b60 initialized for platform METAL (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1775597739.450634 26986495 service.cc:153]   StreamExecutor device (0): Metal, <undefined>
I0000 00:00:1775597739.452854 26986495 mps_client.cc:406] Using Simple allocator.
I0000 00:00:1775597739.452873 26986495 mps_client.cc:384] XLA backend will use up to 11452858368 bytes on device 0 for SimpleAllocator.
Platform 'METAL' is experimental and not all JAX functionality may be correctly supported!
W0000 00:00:1775597740.703649 26986625 mps_client.cc:510] WARNING: JAX Apple GPU support is experimental and not all JAX functionality is correctly supported!
I0000 00:00:1775597740.711510 26986625 service.cc:145] XLA service 0x131fc6de0 initialized for platfor

Metal device set to: Apple M1 Pro

systemMemory: 16.00 GB
maxCacheSize: 5.33 GB

{'backend': 'cpu', 'devices': ['TFRT_CPU_0']}


In [ ]:
from samplers import (  run_ps_rwm_once,
                        run_ps_hmc_once,
                        run_ps_ula_once,
                        run_ps_mala_once,
                        run_ps_nuts_once,)


out_nuts = run_ps_nuts_once(dimension=10, num_particles=32, seed=0)
print("NUTS acceptance path:", out_nuts["acceptance_path"], "NUTS step size path", out_nuts["step_size_path"])
print("HMC acceptance path:", out_hmc["acceptance_path"], "HMC step size path:", out_hmc["step_size_path"])
print("RWM acceptance path:", out_rwm["acceptance_path"], "RWM step size path", out_rwm["step_size_path"])
print("ULA acceptance path:", out_ula["acceptance_path"], "ULA step size path", out_ula["step_size_path"])
print("MALA acceptance path:", out_mala["acceptance_path"], "MALA step size path", out_mala["step_size_path"])

out_hmc = run_ps_hmc_once(dimension=10, num_particles=32, seed=0)
out_rwm = run_ps_rwm_once(dimension=10, num_particles=32, seed=0)
out_ula = run_ps_ula_once(dimension=10, num_particles=32, seed=0)
out_mala = run_ps_mala_once(dimension=10, num_particles=32, seed=0)

NUTS acceptance path: [0.95802742 0.98905438 0.9915821  0.99540871 0.99813789 0.99935573
 0.99935025 0.99827641 0.99521226 0.98956269 0.97168255 0.94273466
 0.8877793  0.86497551] NUTS step size path [0.1        0.13672175 0.18333793 0.23545125 0.29432902 0.36052415
 0.43425646 0.5156253  0.60458666 0.7003991  0.80155897 0.90011871
 0.98675334 1.04156029]


'out_hmc = run_ps_hmc_once(dimension=10, num_particles=32, seed=0)\nout_rwm = run_ps_rwm_once(dimension=10, num_particles=32, seed=0)\nout_ula = run_ps_ula_once(dimension=10, num_particles=32, seed=0)\nout_mala = run_ps_mala_once(dimension=10, num_particles=32, seed=0)\nprint("HMC acceptance path:", out_hmc["acceptance_path"], "HMC step size path:", out_hmc["step_size_path"])\nprint("RWM acceptance path:", out_rwm["acceptance_path"], "RWM step size path", out_rwm["step_size_path"])\nprint("ULA acceptance path:", out_ula["acceptance_path"], "ULA step size path", out_ula["step_size_path"])\nprint("MALA acceptance path:", out_mala["acceptance_path"], "MALA step size path", out_mala["step_size_path"])'